# WC26 — Predicting the Final 8 Knockout Matches

Using **player behavior** and **squad composition** from the 96 completed matches to predict the 8 remaining knockout fixtures (4 QF, 2 SF, 3rd-place, Final).

**As of 2026-07-14:** Group + R32 + R16 complete (96/104). 8 teams alive.
**Approach:** opponent-adjusted attack/defense ratings (goals blended with xG, regularized toward a squad-strength prior) → Poisson goals model → 10k Monte Carlo bracket simulation.

In [1]:
import dataclaw_data
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
DS = 'b3ee5433'
def load(t): return dataclaw_data.get_dataframe(DS, table_name=t)

teams      = load('file_archive_teams_csv')
matches    = load('file_archive_matches_csv')
stages     = load('file_archive_tournament_stages_csv')
lineups    = load('file_archive_match_lineups_csv')
events     = load('file_archive_match_events_csv')
tstats     = load('file_archive_match_team_stats_csv')
pstats     = load('file_archive_player_stats_csv')
squads     = load('file_archive_squads_and_players_csv')
referees   = load('file_archive_referees_csv')

# cast VARCHAR numeric cols in player_stats
for c in ['shots','shots_on_target','average_rating']:
    pstats[c] = pd.to_numeric(pstats[c], errors='coerce')

TEAM = teams.set_index('team_id')['team_name'].to_dict()
CODE = teams.set_index('team_id')['fifa_code'].to_dict()
ELO  = teams.set_index('team_id')['elo_rating'].to_dict()
RANK = teams.set_index('team_id')['fifa_ranking_pre_tournament'].to_dict()

print('rows:', {k: len(v) for k,v in dict(teams=teams,matches=matches,lineups=lineups,events=events,
      tstats=tstats,pstats=pstats,squads=squads,referees=referees).items()})
print('match status:', matches['status'].value_counts().to_dict())
print('player_stats NaN after cast:', pstats[['shots','shots_on_target','average_rating']].isna().sum().to_dict())

rows: {'teams': 48, 'matches': 104, 'lineups': 4992, 'events': 758, 'tstats': 192, 'pstats': 1248, 'squads': 1248, 'referees': 28}
match status: {'Completed': 96, 'Scheduled': 8}
player_stats NaN after cast: {'shots': 1248, 'shots_on_target': 1248, 'average_rating': 1248}


## 1 · Modeling base & bracket
Long team-match table (2 rows per completed match) + reconstructed 8-match bracket. Knockout home/away is bracket seeding only → treated as neutral.

In [2]:
comp = matches[matches.status=='Completed'].copy()

# long team-match table: one row per team per match
def side(df, home=True):
    p = 'home' if home else 'away'; o = 'away' if home else 'home'
    r = pd.DataFrame({
        'match_id': df.match_id, 'stage_id': df.stage_id,
        'team_id': df[f'{p}_team_id'], 'opp_id': df[f'{o}_team_id'],
        'gf': df[f'{p}_score'], 'ga': df[f'{o}_score'],
        'xg_for': df[f'{p}_xg'], 'xg_ag': df[f'{o}_xg'],
        'is_home': 1 if home else 0,
        'pen_for': df[f'{p}_penalty_score'], 'pen_ag': df[f'{o}_penalty_score'],
    })
    return r
tm = pd.concat([side(comp,True), side(comp,False)], ignore_index=True)
# merge team-match shot stats
tm = tm.merge(tstats[['match_id','team_id','possession_pct','total_shots','shots_on_target','corners','fouls']],
              on=['match_id','team_id'], how='left')
tm['is_knockout'] = tm.stage_id >= 2
print("team-match rows:", len(tm), "| completed matches:", comp.shape[0])
print("home advantage (group stage): home gf %.2f vs away gf %.2f" %
      (tm[(tm.stage_id==1)&(tm.is_home==1)].gf.mean(), tm[(tm.stage_id==1)&(tm.is_home==0)].gf.mean()))

# 8 alive teams (R16 winners) and reconstructed bracket
ALIVE = {33:'France',10:'Morocco',36:'Norway',45:'England',
         None:None}  # placeholder, fill below via query result
# team_ids for alive teams from names
name2id = {v:k for k,v in TEAM.items()}
alive_names = ['Argentina','Spain','France','England','Belgium','Morocco','Switzerland','Norway']
alive = {n: name2id[n] for n in alive_names}
print("\nAlive team_ids:", alive)

# remaining bracket (QF98=ESP-BEL, QF100=ARG-SUI inferred from R16 adjacency)
bracket = {
 'QF97': ('France','Morocco'), 'QF98': ('Spain','Belgium'),
 'QF99': ('Norway','England'), 'QF100': ('Argentina','Switzerland'),
 'SF101': ('QF97','QF98'), 'SF102': ('QF99','QF100'),
 'FINAL': ('SF101','SF102'), 'THIRD': ('SF101L','SF102L'),
}
bracket

team-match rows: 192 | completed matches: 96
home advantage (group stage): home gf 1.79 vs away gf 1.19

Alive team_ids: {'Argentina': 37, 'Spain': 29, 'France': 33, 'England': 45, 'Belgium': 25, 'Morocco': 10, 'Switzerland': 8, 'Norway': 36}


{'QF97': ('France', 'Morocco'),
 'QF98': ('Spain', 'Belgium'),
 'QF99': ('Norway', 'England'),
 'QF100': ('Argentina', 'Switzerland'),
 'SF101': ('QF97', 'QF98'),
 'SF102': ('QF99', 'QF100'),
 'FINAL': ('SF101', 'SF102'),
 'THIRD': ('SF101L', 'SF102L')}

## 2 · Squad composition & player-behavior features
Per-team: top-15 market value, age/experience/height profile, positional depth (composition); goals, assists, goal concentration, discipline (behavior). Combined into a **squad-strength prior** used to regularize the on-pitch ratings.

In [3]:
REF = pd.Timestamp('2026-07-14')
sq = squads.copy()
sq['dob'] = pd.to_datetime(sq['date_of_birth'], errors='coerce')
sq['age'] = (REF - sq['dob']).dt.days/365.25
def posgroup(p):
    p=str(p).upper()
    if 'GK' in p or 'KEEP' in p: return 'GK'
    if any(k in p for k in ['CB','LB','RB','DEF','BACK','WB']): return 'DEF'
    if any(k in p for k in ['DM','CM','AM','MID','MF']): return 'MID'
    return 'FWD'
sq['posg'] = sq['position'].map(posgroup)

def squad_feats(g):
    g = g.sort_values('market_value_eur', ascending=False)
    top15 = g.head(15)
    depth = g['posg'].value_counts()
    return pd.Series({
        'squad_val_top15': top15['market_value_eur'].sum(),
        'squad_val_total': g['market_value_eur'].sum(),
        'avg_age': g['age'].mean(),
        'avg_caps': g['caps'].mean(),
        'star_caps': top15['caps'].mean(),
        'avg_height': g['height_cm'].mean(),
        'n_players': len(g),
        'depth_DEF': depth.get('DEF',0), 'depth_MID': depth.get('MID',0), 'depth_FWD': depth.get('FWD',0),
    })
scomp = sq.groupby('team_id').apply(squad_feats, include_groups=False)

# player behavior from player_stats (team-level)
def behav(g):
    tg = g['goals'].sum()
    top = g.nlargest(1,'goals')
    return pd.Series({
        'goals_total': tg,
        'assists_total': g['assists'].sum(),
        'top_scorer': top['player_name'].iloc[0] if len(top) else None,
        'top_scorer_goals': int(top['goals'].iloc[0]) if len(top) else 0,
        'top1_share': (top['goals'].iloc[0]/tg) if tg>0 else 0,      # reliance on one striker
        'yellows': g['yellow_cards'].sum(), 'reds': g['red_cards'].sum(),
        'n_scorers': int((g['goals']>0).sum()),
    })
pbeh = pstats.groupby('team_id').apply(behav, include_groups=False)

# matches played per team (for shrinkage) + goals/xg aggregates
mp = tm.groupby('team_id').agg(mp=('match_id','count'),
        gf=('gf','sum'), ga=('ga','sum'), xgf=('xg_for','sum'), xga=('xg_ag','sum')).reset_index()

feat = (mp.merge(scomp.reset_index(), on='team_id', how='left')
          .merge(pbeh.reset_index(), on='team_id', how='left'))
feat['team'] = feat.team_id.map(TEAM); feat['elo']=feat.team_id.map(ELO); feat['rank']=feat.team_id.map(RANK)
feat['disc_per_match'] = (feat.yellows + 3*feat.reds)/feat.mp

# strength prior: z-score elo, log top15 value, star caps  -> composite
import numpy as np
def z(s): return (s-s.mean())/s.std(ddof=0)
feat['prior_raw'] = 0.55*z(feat.elo) + 0.30*z(np.log(feat.squad_val_top15)) + 0.15*z(feat.star_caps)
feat['prior_z'] = z(feat.prior_raw)

cols=['team','elo','rank','squad_val_top15','avg_age','star_caps','goals_total','top_scorer','top_scorer_goals','top1_share','disc_per_match','prior_z']
print(feat[feat.team.isin(alive_names)].sort_values('prior_z',ascending=False)[cols].round(2).to_string(index=False))

       team  elo  rank  squad_val_top15  avg_age  star_caps  goals_total           top_scorer  top_scorer_goals  top1_share  disc_per_match  prior_z
      Spain 2120     2     1572399628.0    26.84      31.67            8      Mikel Oyarzabal                 4        0.50             0.6     2.02
  Argentina 2150     1      937539444.0    29.21      33.13           13  Lionel Andrés Messi                 8        0.62             0.4     1.98
     France 2100     3     1313204469.0    27.09      30.27           14        Kylian Mbappe                 7        0.50             0.2     1.85
    England 2050     4     1102913351.0    27.36      28.53           11    Harry Edward Kane                 6        0.55             2.0     1.54
    Belgium 1950     9      485117492.0    27.70      29.40           12        Romelu Lukaku                 3        0.25             1.4     0.84
    Morocco 1920     7      413054154.0    26.69      28.00           10       Ismael Saibari             

## 3 · Attack/defense ratings + holdout validation
Iterative opponent-adjusted attack/defense factors on **blended goals** (0.6·xG + 0.4·actual, xG is more stable), shrunk toward the squad-strength prior (weight = mp/(mp+5)). **Validation:** fit on group stage only, predict the 24 completed knockouts.

In [4]:
from scipy.stats import poisson
PRIORZ = feat.set_index('team_id')['prior_z'].to_dict()
ALL_TEAMS = feat.team_id.tolist()

def fit_ratings(tm_fit, w_xg=0.6, K=5.0, beta=0.16, n_iter=200):
    d = tm_fit.copy()
    d['gs_for'] = w_xg*d.xg_for + (1-w_xg)*d.gf
    d['gs_ag']  = w_xg*d.xg_ag + (1-w_xg)*d.ga
    mu = d['gs_for'].mean()
    mpd = d.groupby('team_id').size().to_dict()
    atk = {t:1.0 for t in ALL_TEAMS}; dfn = {t:1.0 for t in ALL_TEAMS}; gamma=1.3
    rows = list(d.itertuples())
    for _ in range(n_iter):
        na={t:0. for t in ALL_TEAMS}; da={t:1e-9 for t in ALL_TEAMS}
        nd={t:0. for t in ALL_TEAMS}; dd={t:1e-9 for t in ALL_TEAMS}
        ghn=0.; ghd=1e-9
        for r in rows:
            i,j,h = r.team_id, r.opp_id, r.is_home
            na[i]+=r.gs_for;  da[i]+=mu*dfn[j]*(gamma if h else 1)
            nd[i]+=r.gs_ag;   dd[i]+=mu*atk[j]*(gamma if not h else 1)
            if h: ghn+=r.gs_for; ghd+=mu*atk[i]*dfn[j]
        for t in ALL_TEAMS:
            if t in mpd: atk[t]=na[t]/da[t]; dfn[t]=nd[t]/dd[t]
        ma=np.mean([atk[t] for t in ALL_TEAMS]); md=np.mean([dfn[t] for t in ALL_TEAMS])
        for t in ALL_TEAMS: atk[t]/=ma; dfn[t]/=md
        gamma=ghn/ghd
    # shrink toward prior
    af={}; df_={}
    for t in ALL_TEAMS:
        n=mpd.get(t,0); w=n/(n+K)
        af[t]=np.exp(w*np.log(atk[t]) + (1-w)*( beta*PRIORZ[t]))
        df_[t]=np.exp(w*np.log(dfn[t]) + (1-w)*(-beta*PRIORZ[t]))
    return dict(mu=mu, atk=af, dfn=df_, gamma=gamma)

def match_probs(lamA,lamB,mx=12):
    pa=poisson.pmf(np.arange(mx+1),lamA); pb=poisson.pmf(np.arange(mx+1),lamB)
    M=np.outer(pa,pb)
    return np.tril(M,-1).sum(), np.trace(M), np.triu(M,1).sum()   # pA, draw, pB

def strength(R,t): return np.log(R['atk'][t]) - np.log(R['dfn'][t])
def pen_p(R,A,B): return 1/(1+np.exp(-0.4*(strength(R,A)-strength(R,B))))   # mild favorite edge

def adv_prob(R,A,B):   # prob A advances in a knockout (neutral)
    lamA=R['mu']*R['atk'][A]*R['dfn'][B]; lamB=R['mu']*R['atk'][B]*R['dfn'][A]
    pA,pd,pB=match_probs(lamA,lamB)
    return pA + pd*pen_p(R,A,B), (lamA,lamB,pA,pd,pB)

# ---- VALIDATION: fit on group stage, predict completed knockouts ----
Rg = fit_ratings(tm[tm.stage_id==1])
ko = comp[comp.stage_id>=2].copy()
def home_advanced(r):
    if r.home_score!=r.away_score: return int(r.home_score>r.away_score)
    return int(r.home_penalty_score>r.away_penalty_score)
y, p = [], []
for r in ko.itertuples():
    p_home,_ = adv_prob(Rg, r.home_team_id, r.away_team_id)
    p.append(p_home); y.append(home_advanced(r))
y=np.array(y); p=np.clip(np.array(p),1e-6,1-1e-6)
acc=((p>0.5)==(y==1)).mean(); ll=-(y*np.log(p)+(1-y)*np.log(1-p)).mean(); brier=((p-y)**2).mean()
base=-(y.mean()*np.log(y.mean())+(1-y.mean())*np.log(1-y.mean()))
print(f"Holdout on {len(y)} completed knockouts (group-only ratings):")
print(f"  accuracy={acc:.3f}   log-loss={ll:.3f} (baseline {base:.3f})   brier={brier:.3f}")
# calibration in 3 buckets
dfc=pd.DataFrame({'p':p,'y':y}); dfc['bk']=pd.cut(dfc.p,[0,.4,.6,1])
print(dfc.groupby('bk',observed=True).agg(pred=('p','mean'),actual=('y','mean'),n=('y','size')).round(2).to_string())

Holdout on 24 completed knockouts (group-only ratings):
  accuracy=0.667   log-loss=0.544 (baseline 0.690)   brier=0.181
            pred  actual   n
bk                          
(0.0, 0.4]  0.31    0.00   3
(0.4, 0.6]  0.53    0.36  11
(0.6, 1.0]  0.73    0.90  10


## 4 · Final ratings, QF predictions & 10k Monte Carlo
Refit on all 96 completed matches. All four QF matchups are now known → direct analytic predictions. SF/Final/3rd depend on results → 10k bracket simulations (extra time + penalty shootouts) for advancement and title odds.

In [5]:
R = fit_ratings(tm)   # final ratings on all 96 completed matches

# alive team rating table
rt = pd.DataFrame({'team_id':list(alive.values())})
rt['team']=rt.team_id.map(TEAM)
rt['atk']=rt.team_id.map(R['atk']); rt['dfn']=rt.team_id.map(R['dfn'])
rt['strength']=rt.team_id.apply(lambda t: strength(R,t))
rt['xg_for_vs_avg']=R['mu']*rt.atk; rt['xg_ag_vs_avg']=R['mu']*rt.dfn
rt=rt.sort_values('strength',ascending=False).reset_index(drop=True)
print("FINAL TEAM RATINGS (alive 8), sorted by overall strength:")
print(rt[['team','atk','dfn','strength','xg_for_vs_avg','xg_ag_vs_avg']].round(3).to_string(index=False))

def scoreline(lamA,lamB,mx=8):
    pa=poisson.pmf(np.arange(mx+1),lamA); pb=poisson.pmf(np.arange(mx+1),lamB)
    M=np.outer(pa,pb); i,j=np.unravel_index(M.argmax(),M.shape); return i,j,M.max()

QF=[('QF97','France','Morocco'),('QF98','Spain','Belgium'),('QF99','Norway','England'),('QF100','Argentina','Switzerland')]
print("\nQUARTER-FINAL DIRECT PREDICTIONS (neutral venue):")
qf_rows=[]
for tag,A,B in QF:
    a,b=alive[A],alive[B]
    lamA=R['mu']*R['atk'][a]*R['dfn'][b]; lamB=R['mu']*R['atk'][b]*R['dfn'][a]
    pA,pd_,pB=match_probs(lamA,lamB)
    advA=pA+pd_*pen_p(R,a,b)
    si,sj,_=scoreline(lamA,lamB)
    qf_rows.append(dict(match=tag,A=A,B=B,padvA=advA,padvB=1-advA,lamA=lamA,lamB=lamB,ml=f"{si}-{sj}"))
    print(f"  {A} vs {B}: {A} {advA*100:4.1f}% | {B} {(1-advA)*100:4.1f}%  (xG {lamA:.2f}-{lamB:.2f}, likely {si}-{sj})")
qf_df=pd.DataFrame(qf_rows)

# ---- Monte Carlo bracket ----
rng=np.random.default_rng(7)
def sim_ko(A,B):
    lamA=R['mu']*R['atk'][A]*R['dfn'][B]; lamB=R['mu']*R['atk'][B]*R['dfn'][A]
    gA=rng.poisson(lamA); gB=rng.poisson(lamB)
    if gA==gB:
        gA+=rng.poisson(lamA/3); gB+=rng.poisson(lamB/3)   # extra time
        if gA==gB:
            return (A if rng.random()<pen_p(R,A,B) else B)
    return A if gA>gB else B
def other(pair,w): return pair[0] if w==pair[1] else pair[1]

N=10000
from collections import Counter
champ=Counter(); finalist=Counter(); sf=Counter(); third=Counter(); finals=Counter()
for _ in range(N):
    w97=sim_ko(alive['France'],alive['Morocco']); w98=sim_ko(alive['Spain'],alive['Belgium'])
    w99=sim_ko(alive['Norway'],alive['England']); w100=sim_ko(alive['Argentina'],alive['Switzerland'])
    for t in (w97,w98,w99,w100): sf[t]+=1
    s1=(w97,w98); s2=(w99,w100)
    f1=sim_ko(*s1); f2=sim_ko(*s2)
    finalist[f1]+=1; finalist[f2]+=1
    l1=other(s1,f1); l2=other(s2,f2)
    ch=sim_ko(f1,f2); champ[ch]+=1
    finals[tuple(sorted((TEAM[f1],TEAM[f2])))]+=1
    third[sim_ko(l1,l2)]+=1

ids={v:k for k,v in alive.items()}
proj=pd.DataFrame({'team':list(alive.keys())})
proj['team_id']=proj.team.map(alive)
proj['reach_SF']=proj.team_id.map(lambda t: sf[t]/N)
proj['reach_Final']=proj.team_id.map(lambda t: finalist[t]/N)
proj['champion']=proj.team_id.map(lambda t: champ[t]/N)
proj['third_place']=proj.team_id.map(lambda t: third[t]/N)
proj['podium']=proj.champion+proj.team_id.map(lambda t: finalist[t]/N - champ[t]/N)+proj.third_place
proj=proj.sort_values('champion',ascending=False).reset_index(drop=True)
print("\nMONTE-CARLO TOURNAMENT PROJECTION (10k sims):")
print(proj[['team','reach_SF','reach_Final','champion','third_place']].round(3).to_string(index=False))
print("\nMost likely FINAL matchups:")
for m,c in finals.most_common(5): print(f"  {m[0]} vs {m[1]}: {c/N*100:.1f}%")

FINAL TEAM RATINGS (alive 8), sorted by overall strength:
       team   atk   dfn  strength  xg_for_vs_avg  xg_ag_vs_avg
      Spain 1.394 0.404     1.239          1.926         0.558
     France 1.586 0.481     1.194          2.191         0.664
    Morocco 1.407 0.624     0.814          1.944         0.861
     Norway 1.532 0.838     0.603          2.117         1.158
  Argentina 1.358 0.743     0.603          1.876         1.026
    Belgium 1.329 0.731     0.597          1.836         1.010
    England 1.384 0.913     0.416          1.913         1.261
Switzerland 1.065 0.843     0.234          1.471         1.165

QUARTER-FINAL DIRECT PREDICTIONS (neutral venue):
  France vs Morocco: France 61.6% | Morocco 38.4%  (xG 1.37-0.93, likely 1-0)
  Spain vs Belgium: Spain 68.4% | Belgium 31.6%  (xG 1.41-0.74, likely 1-0)
  Norway vs England: Norway 57.1% | England 42.9%  (xG 1.93-1.60, likely 1-1)
  Argentina vs Switzerland: Argentina 62.1% | Switzerland 37.9%  (xG 1.58-1.09, likely 1-1)


## 5 · Visualizations

In [6]:
import os, matplotlib as mpl
mpl.rcParams.update({'font.size':11,'axes.grid':True,'grid.alpha':.25,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':130})
CWD=os.getcwd(); print("saving to", CWD)
COL={'Spain':'#E4572E','France':'#2E4A8B','Argentina':'#4FA9E0','Norway':'#7B2D40',
     'Morocco':'#148F55','England':'#C0392B','Belgium':'#E1A700','Switzerland':'#7D3C98'}
paths={}

# --- FIG 1: Title race (champion % + reach-final %) ---
d=proj.sort_values('champion')
fig,ax=plt.subplots(figsize=(8.6,5))
ax.barh(d.team, d.reach_Final*100, color='#dfe3ea', label='Reach Final', zorder=1)
ax.barh(d.team, d.champion*100, color=[COL[t] for t in d.team], label='Win title', zorder=2)
for y,(c,f) in enumerate(zip(d.champion,d.reach_Final)):
    ax.text(c*100+0.4, y, f"{c*100:.0f}%", va='center', fontweight='bold', fontsize=10)
    ax.text(f*100+0.4, y, f"{f*100:.0f}%", va='center', color='#7a8190', fontsize=8.5)
ax.set_xlabel('Probability (%)'); ax.set_title('WC26 Title Race — champion vs reach-final odds (10k sims)',fontweight='bold')
ax.legend(loc='lower right',frameon=False); fig.tight_layout()
p=f'{CWD}/wc26_f8_title_race.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['title']=p

# --- FIG 2: Strength map (attack vs defense) ---
fig,ax=plt.subplots(figsize=(8.6,6))
for _,r in rt.iterrows():
    x,yv=r.xg_for_vs_avg, r.xg_ag_vs_avg
    ax.scatter(x,yv,s=300+proj.set_index('team').champion[r.team]*4500,color=COL[r.team],alpha=.85,edgecolor='white',lw=1.5,zorder=3)
    ax.annotate(r.team,(x,yv),xytext=(0,-4),textcoords='offset points',ha='center',va='top',fontsize=9,fontweight='bold')
ax.axvline(R['mu'],color='#aaa',ls='--',lw=.8); ax.axhline(R['mu'],color='#aaa',ls='--',lw=.8)
ax.invert_yaxis()  # up = fewer goals conceded = better defense
ax.set_xlabel('Attack →  expected goals for (per match)'); ax.set_ylabel('← Defense  expected goals against')
ax.set_title('Team Strength Map — bubble size = title odds\n(top-right = elite both ends)',fontweight='bold')
ax.text(.99,.02,'dashed = tournament average',transform=ax.transAxes,ha='right',color='#999',fontsize=8)
fig.tight_layout(); p=f'{CWD}/wc26_f8_strength_map.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['map']=p

# --- FIG 3: Quarter-final win probabilities ---
fig,ax=plt.subplots(figsize=(8.6,4.6))
qd=qf_df.iloc[::-1].reset_index(drop=True)
for y,r in qd.iterrows():
    ax.barh(y, -r.padvA*100, color=COL[r.A]); ax.barh(y, r.padvB*100, color=COL[r.B])
    ax.text(-r.padvA*100-1, y, f"{r.A} {r.padvA*100:.0f}%", va='center', ha='right', fontsize=9.5, fontweight='bold')
    ax.text(r.padvB*100+1, y, f"{r.B} {r.padvB*100:.0f}%", va='center', ha='left', fontsize=9.5, fontweight='bold')
    ax.text(0,y+0.34,f"xG {r.lamA:.2f}–{r.lamB:.2f} · likely {r.ml}",va='bottom',ha='center',fontsize=7.8,color='#666')
ax.axvline(0,color='#333',lw=1); ax.set_xlim(-100,100); ax.set_yticks([]); ax.set_xlabel('← advance %      advance % →')
ax.set_title('Quarter-final Predictions (all 4 matchups known)',fontweight='bold'); ax.grid(False)
fig.tight_layout(); p=f'{CWD}/wc26_f8_qf.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['qf']=p

# --- FIG 4: Advancement funnel ---
fig,ax=plt.subplots(figsize=(8.6,5))
d=proj.sort_values('champion',ascending=False); x=np.arange(len(d)); w=.26
ax.bar(x-w, d.reach_SF*100, w, label='Reach SF', color='#b9c2d0')
ax.bar(x,   d.reach_Final*100, w, label='Reach Final', color='#6f7f99')
ax.bar(x+w, d.champion*100, w, label='Champion', color=[COL[t] for t in d.team])
ax.set_xticks(x); ax.set_xticklabels(d.team,rotation=25,ha='right'); ax.set_ylabel('Probability (%)')
ax.set_title('Advancement Funnel by Team',fontweight='bold'); ax.legend(frameon=False)
fig.tight_layout(); p=f'{CWD}/wc26_f8_funnel.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['funnel']=p

# --- FIG 5: Player behavior & squad composition ---
fb=feat[feat.team.isin(alive_names)].copy().sort_values('goals_total')
fig,(a1,a2)=plt.subplots(1,2,figsize=(11.5,5))
a1.barh(fb.team, fb.goals_total, color=[COL[t] for t in fb.team])
for y,(g,ts,tg,sh) in enumerate(zip(fb.goals_total,fb.top_scorer,fb.top_scorer_goals,fb.top1_share)):
    a1.text(g+0.2,y,f"{str(ts).split()[-1]} {tg} ({sh*100:.0f}%)",va='center',fontsize=8.2,color='#444')
a1.set_title('Goals scored & top-scorer reliance',fontweight='bold'); a1.set_xlabel('Team goals (tournament)')
a1.set_xlim(0,fb.goals_total.max()+6)
sc=a2.scatter(fb.squad_val_top15/1e9, fb.disc_per_match, s=180, color=[COL[t] for t in fb.team], edgecolor='white', lw=1.4)
for _,r in fb.iterrows(): a2.annotate(r.team,(r.squad_val_top15/1e9,r.disc_per_match),xytext=(5,3),textcoords='offset points',fontsize=8.5,fontweight='bold')
a2.set_xlabel('Top-15 squad market value (€B)'); a2.set_ylabel('Discipline risk (cards/match, red=×3)')
a2.set_title('Squad value vs discipline risk',fontweight='bold')
fig.suptitle('Player Behavior & Squad Composition — the 8 live teams',fontweight='bold',y=1.02)
fig.tight_layout(); p=f'{CWD}/wc26_f8_players.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['players']=p

print("saved:", list(paths.values()))

saving to /Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b


saved: ['/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_title_race.png', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_strength_map.png', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_qf.png', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_funnel.png', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_players.png']


In [7]:
import plotly.graph_objects as go, json
FONT=dict(family="Inter, Segoe UI, sans-serif")
def savef(fig, name):
    fig.update_layout(font=FONT, template='plotly_white', margin=dict(l=60,r=30,t=60,b=50))
    path=f'{CWD}/{name}.json'
    open(path,'w').write(fig.to_json())
    return path
figpaths={}
order=proj.sort_values('champion')

# 1. Title race
f=go.Figure()
f.add_bar(y=order.team, x=(order.reach_Final*100).round(1), orientation='h', name='Reach Final',
          marker_color='#dfe3ea', hovertemplate='%{y}: %{x}% reach final<extra></extra>')
f.add_bar(y=order.team, x=(order.champion*100).round(1), orientation='h', name='Win title',
          marker_color=[COL[t] for t in order.team], text=[f"{c*100:.0f}%" for c in order.champion],
          textposition='outside', hovertemplate='%{y}: %{x}% champion<extra></extra>')
f.update_layout(barmode='overlay', title='Title Race — champion vs reach-final odds (10k sims)',
                xaxis_title='Probability (%)', legend=dict(orientation='h',y=-0.15))
figpaths['title']=savef(f,'fig_title')

# 2. Quarter-finals diverging
f=go.Figure()
qd=qf_df.iloc[::-1]
f.add_bar(y=qd.A, x=-(qd.padvA*100).round(1), orientation='h', marker_color=[COL[t] for t in qd.A],
          text=[f"{a} {p*100:.0f}%" for a,p in zip(qd.A,qd.padvA)], textposition='outside',
          hovertemplate='%{text}<extra></extra>', showlegend=False)
f.add_bar(y=qd.B, x=(qd.padvB*100).round(1), orientation='h', marker_color=[COL[t] for t in qd.B],
          text=[f"{b} {p*100:.0f}%" for b,p in zip(qd.B,qd.padvB)], textposition='outside',
          hovertemplate='%{text}<extra></extra>', showlegend=False)
f.update_layout(barmode='relative', title='Quarter-final Predictions — advance probability (all 4 matchups known)',
                xaxis=dict(title='← advance %      advance % →', range=[-115,115]),
                yaxis=dict(showticklabels=False))
for i,(_,r) in enumerate(qd.iterrows()):
    f.add_annotation(x=0, y=r.A, yshift=15, text=f"xG {r.lamA:.2f}–{r.lamB:.2f} · likely {r.ml}",
                     showarrow=False, font=dict(size=10,color='#777'))
figpaths['qf']=savef(f,'fig_qf')

# 3. Advancement funnel
d=proj.sort_values('champion',ascending=False)
f=go.Figure()
f.add_bar(x=d.team, y=(d.reach_SF*100).round(1), name='Reach SF', marker_color='#b9c2d0')
f.add_bar(x=d.team, y=(d.reach_Final*100).round(1), name='Reach Final', marker_color='#6f7f99')
f.add_bar(x=d.team, y=(d.champion*100).round(1), name='Champion', marker_color=[COL[t] for t in d.team])
f.update_layout(barmode='group', title='Advancement Funnel by Team', yaxis_title='Probability (%)',
                legend=dict(orientation='h',y=-0.2))
figpaths['funnel']=savef(f,'fig_funnel')

# 4. Strength map
f=go.Figure()
f.add_scatter(x=rt.xg_for_vs_avg, y=rt.xg_ag_vs_avg, mode='markers+text',
              text=rt.team, textposition='bottom center',
              marker=dict(size=[18+proj.set_index('team').champion[t]*160 for t in rt.team],
                          color=[COL[t] for t in rt.team], line=dict(color='white',width=1.5)),
              hovertemplate='%{text}<br>xG for %{x:.2f} · xG against %{y:.2f}<extra></extra>')
f.add_vline(x=R['mu'], line=dict(color='#bbb',dash='dash')); f.add_hline(y=R['mu'], line=dict(color='#bbb',dash='dash'))
f.update_layout(title='Team Strength Map — bubble = title odds (top-right = elite both ends)',
                xaxis_title='Attack →  expected goals for', yaxis=dict(title='← Defense  expected goals against', autorange='reversed'))
figpaths['map']=savef(f,'fig_map')

print(json.dumps({k:os.path.getsize(v) for k,v in figpaths.items()}, indent=0))
print(list(figpaths.values()))

{
"title": 7858,
"qf": 8306,
"funnel": 7813,
"map": 8126
}
['/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/fig_title.json', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/fig_qf.json', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/fig_funnel.json', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/fig_map.json']
